# STM32F429 CAN (Controller Area Network) Tutorial for Beginners

This tutorial provides a comprehensive guide to understanding and using the CAN bus peripheral on STM32F429 microcontrollers. We'll cover everything from basic CAN communication to advanced features like filtering, error handling, and multi-node networks.

## Table of Contents
1. [Introduction to CAN Bus](#introduction)
2. [STM32F429 CAN Overview](#can-overview)
3. [Hardware Setup](#hardware-setup)
4. [Software Dependencies](#software-setup)
5. [Basic CAN Communication](#basic-communication)
6. [CAN Frame Types](#frame-types)
7. [Baud Rate Configuration](#baud-rate)
8. [Message Filtering](#filtering)
9. [Error Handling](#error-handling)
10. [Interrupt-Driven CAN](#interrupts)
11. [Multi-Node Networks](#multi-node)
12. [CAN Bus Monitoring](#monitoring)
13. [Troubleshooting](#troubleshooting)

<a id='introduction'></a>
## 1. Introduction to CAN Bus

CAN (Controller Area Network) is a robust serial communication protocol originally developed for automotive applications. It provides reliable communication in harsh environments with excellent error detection and fault tolerance.

### Key CAN Concepts:
- **Multi-Master**: Any node can initiate communication
- **Message-Based**: Data is sent in messages, not to specific addresses
- **Priority-Based**: Messages have priority levels (lower ID = higher priority)
- **Error Detection**: Multiple error detection mechanisms
- **Fault Tolerance**: Continues operation despite faults
- **Real-Time**: Deterministic message delivery

### CAN Message Structure:
```
┌─────────┬─────────────┬──────────────┬─────────┬─────────┐
│ SOF     │ Arbitration │ Control      │ Data    │ CRC     │
│ (1 bit) │ (11/29 bit) │ (6 bit)      │ (0-64)  │ (15 bit)│
└─────────┴─────────────┴──────────────┴─────────┴─────────┘
                          ┌─────────┬─────────┐
                          │ ACK     │ EOF     │
                          │ (2 bit) │ (7 bit) │
                          └─────────┴─────────┘
```

### CAN Bus States:
- **Active Error**: Normal operation
- **Passive Error**: Can receive but transmission restricted
- **Bus Off**: Disconnected from bus due to excessive errors

### CAN Register Map (Reference Manual)

The STM32F429 CAN peripheral uses several registers for configuration:

| Register | Address Offset | Description |
|----------|----------------|-------------|
| CAN_MCR  | 0x000 | Master Control Register |
| CAN_MSR  | 0x004 | Master Status Register |
| CAN_TSR  | 0x008 | Transmit Status Register |
| CAN_RF0R | 0x00C | Receive FIFO 0 Register |
| CAN_RF1R | 0x010 | Receive FIFO 1 Register |
| CAN_IER  | 0x014 | Interrupt Enable Register |
| CAN_ESR  | 0x018 | Error Status Register |
| CAN_BTR  | 0x01C | Bit Timing Register |
| CAN_TI0R | 0x180 | TX Mailbox Identifier Register |
| CAN_TDT0R| 0x184 | TX Mailbox Data Length Register |
| CAN_TDL0R| 0x188 | TX Mailbox Data Low Register |
| CAN_TDH0R| 0x18C | TX Mailbox Data High Register |
| CAN_RI0R | 0x1B0 | RX FIFO Identifier Register |
| CAN_RDT0R| 0x1B4 | RX FIFO Data Length Register |
| CAN_RDL0R| 0x1B8 | RX FIFO Data Low Register |
| CAN_RDH0R| 0x1BC | RX FIFO Data High Register |

### Key CAN Register Configurations:

#### CAN_MCR (Master Control Register):
- **Bit 0**: INRQ - Initialization Request
- **Bit 1**: SLEEP - Sleep Mode Request
- **Bit 4**: TXFP - Transmit FIFO Priority
- **Bit 5**: RFLM - Receive FIFO Locked Mode
- **Bit 6**: NART - No Automatic Retransmission
- **Bit 7**: AWUM - Automatic Wakeup Mode
- **Bit 8**: ABOM - Automatic Bus-Off Management
- **Bit 9**: TTCM - Time Triggered Communication Mode

#### CAN_BTR (Bit Timing Register):
- **Bits 9-0**: BRP - Baud Rate Prescaler
- **Bits 19-16**: TS1 - Time Segment 1
- **Bits 22-20**: TS2 - Time Segment 2
- **Bits 25-24**: SJW - Resynchronization Jump Width

#### CAN_IER (Interrupt Enable Register):
- **Bit 0**: TMEIE - Transmit Mailbox Empty Interrupt
- **Bit 1**: FMPIE0 - FIFO 0 Message Pending Interrupt
- **Bit 2**: FFIE0 - FIFO 0 Full Interrupt
- **Bit 3**: FOVIE0 - FIFO 0 Overrun Interrupt
- **Bit 4**: FMPIE1 - FIFO 1 Message Pending Interrupt
- **Bit 8**: EWGIE - Error Warning Interrupt
- **Bit 9**: EPVIE - Error Passive Interrupt
- **Bit 10**: BOFIE - Bus-Off Interrupt
- **Bit 11**: LECIE - Last Error Code Interrupt

<a id='can-overview'></a>
## 2. STM32F429 CAN Overview

The STM32F429 features a CAN controller that supports both CAN 2.0A (standard) and CAN 2.0B (extended) protocols with advanced features for industrial and automotive applications.

### CAN Features:
- **Dual CAN Controllers**: CAN1 and CAN2 (master-slave relationship)
- **32-bit Filters**: 14 configurable filter banks
- **3 Transmit Mailboxes**: Priority-based transmission
- **2 Receive FIFOs**: Separate FIFO for each receive channel
- **Time Triggered Communication**: For real-time applications
- **Listen-Only Mode**: Bus monitoring without transmission
- **Loopback Mode**: Internal testing without external bus

### CAN Operating Modes:
- **Normal Mode**: Standard CAN operation
- **Loopback Mode**: Internal loopback for testing
- **Silent Mode**: Listen-only for bus monitoring
- **Silent-Loopback Mode**: Combined silent and loopback

### Message Priority:
- **Identifier-Based**: Lower ID = Higher priority
- **TX Mailbox Priority**: Configurable mailbox priority
- **FIFO Priority**: First-in-first-out or identifier-based

### CAN Frame Formats:
- **Standard Frame**: 11-bit identifier (CAN 2.0A)
- **Extended Frame**: 29-bit identifier (CAN 2.0B)
- **Data Frame**: Normal data transmission
- **Remote Frame**: Request for data transmission
- **Error Frame**: Error signaling
- **Overload Frame**: Inter-frame spacing

### Bit Timing:
- **Time Quantum**: Basic unit of time
- **Bit Segments**: Sync, Propagation, Phase 1, Phase 2
- **Sample Point**: Where bit is sampled (typically 75-87.5%)
- **Synchronization**: Hard sync and resync jump width

### Error Detection:
- **Bit Monitoring**: Transmitter monitors its own bits
- **Bit Stuffing**: Error detection through bit stuffing
- **Frame Check**: CRC, ACK, and EOF checks
- **Error Counters**: Transmit and receive error counting

<a id='hardware-setup'></a>
## 3. Hardware Setup

### Required Components:
- STM32F429 Discovery board
- CAN transceiver (MCP2551, TJA1050, etc.)
- CAN bus termination resistors (120Ω)
- CAN bus cable
- Optional: Additional CAN nodes for testing

### CAN Interface Pin Configuration:
```
CAN1_RX  -> PD0   (CAN1 Receive)
CAN1_TX  -> PD1   (CAN1 Transmit)
CAN2_RX  -> PB5   (CAN2 Receive) 
CAN2_TX  -> PB6   (CAN2 Transmit)
```

### CAN Transceiver Connection:
```
STM32 CAN_TX ──┐
                │
                ├─ CANH (High)
CAN Transceiver ├─ CANL (Low)
                │
STM32 CAN_RX ──┘
```

### Bus Termination:
```
CANH ───┬─── 120Ω ──── CANH (other nodes)
        │
       120Ω
        │
CANL ───┴─── 120Ω ──── CANL (other nodes)
```

### Power Supply:
- **CAN Transceiver**: Typically 5V supply
- **Bus Bias**: Some transceivers provide bus biasing
- **Common Ground**: All nodes must share common ground
- **Isolation**: Optional galvanic isolation for noise immunity

### Signal Integrity:
- **Stub Length**: Keep node connections short (<0.3m)
- **Total Length**: Bus length depends on baud rate
- **Termination**: Exactly two 120Ω terminators
- **Impedance**: 120Ω characteristic impedance

### Multi-Node Setup:
```c
// Node 1: Engine Control Unit (ECU)
#define NODE_ID 0x100

// Node 2: Transmission Control Unit (TCU)  
#define NODE_ID 0x200

// Node 3: Body Control Module (BCM)
#define NODE_ID 0x300

// All connected to same CAN bus
// Each node has unique message IDs
```

### Diagnostic Connections:
- **CANH/CANL**: Bus signals for oscilloscope
- **GND**: Reference ground
- **VCC**: Power supply monitoring
- **ERR**: Error output from transceiver

<a id='software-setup'></a>
## 4. Software Dependencies

### Required Libraries:
```c
#include "can.h"           // CAN driver
#include "stm32f4xx_hal.h" // HAL library
#include "stm32f4xx_hal_can.h" // CAN peripheral
#include "stm32f4xx_hal_gpio.h" // GPIO for CAN pins
#include "stm32f4xx_hal_rcc.h" // Clock configuration
```

### CAN Configuration Structure:
```c
typedef struct {
    CAN_OperatingMode mode;                // Operating mode
    uint32_t baud_rate;                    // Baud rate in bps
    bool auto_retransmission;              // Auto retransmission enable
    bool auto_bus_off_recovery;            // Auto bus-off recovery enable
    bool time_triggered_comm;              // Time-triggered communication mode
    uint32_t prescaler;                    // Time quanta prescaler
    uint32_t sync_jump_width;              // Synchronization jump width
    uint32_t time_segment_1;               // Time segment 1
    uint32_t time_segment_2;               // Time segment 2
} CAN_Config;
```

### CAN Frame Structure:
```c
typedef struct {
    uint32_t id;                           // CAN ID (11-bit or 29-bit)
    CAN_FrameType frame_type;              // Standard or Extended frame
    uint8_t data_length;                   // Data length (0-8 bytes)
    uint8_t data[CAN_MAX_DATA_LENGTH];     // Data payload
    bool remote_request;                   // Remote Transmission Request
    uint32_t timestamp;                    // Timestamp (if supported)
} CAN_Frame;
```

### Initialization Sequence:
1. **Enable CAN clock** (RCC_APB1ENR)
2. **Configure GPIO pins** for CAN RX/TX
3. **Initialize CAN peripheral** with configuration
4. **Configure filters** for message acceptance
5. **Start CAN communication**
6. **Enable interrupts** (optional)

### Clock Configuration:
- **CAN Clock**: APB1 clock (up to 42MHz)
- **Prescaler**: Divides CAN clock for bit timing
- **Time Quanta**: Basic unit for bit timing
- **Baud Rate**: Derived from clock and bit timing

### Filter Configuration:
- **Filter Banks**: 14 banks for CAN1, 2 for CAN2
- **Filter Modes**: Mask mode or list mode
- **Filter Scales**: 16-bit or 32-bit filters
- **Identifier Types**: Standard or extended IDs

<a id='basic-communication'></a>
## 5. Basic CAN Communication

### Simple CAN Initialization:
```c
#include "can.h"

void can_example_basic_init(void) {
    // Configure CAN for 500kbps normal operation
    CAN_Config config = {
        .mode = CAN_OP_MODE_NORMAL,
        .baud_rate = CAN_BAUD_500KBPS,
        .auto_retransmission = true,
        .auto_bus_off_recovery = true,
        .time_triggered_comm = false
    };
    
    // Initialize CAN
    HAL_StatusTypeDef status = CAN_Init(&config);
    if (status != HAL_OK) {
        printf("CAN initialization failed: %d\n", status);
        return;
    }
    
    printf("CAN initialized successfully at 500kbps\n");
}
```

### Basic Message Transmission:
```c
void can_example_transmit(void) {
    // Create a CAN frame
    CAN_Frame tx_frame = {
        .id = 0x123,                    // Message ID
        .frame_type = CAN_FRAME_STANDARD, // Standard frame
        .data_length = 8,               // 8 bytes of data
        .remote_request = false         // Data frame
    };
    
    // Fill data payload
    tx_frame.data[0] = 0xAA;
    tx_frame.data[1] = 0xBB;
    tx_frame.data[2] = 0xCC;
    tx_frame.data[3] = 0xDD;
    tx_frame.data[4] = 0xEE;
    tx_frame.data[5] = 0xFF;
    tx_frame.data[6] = 0x11;
    tx_frame.data[7] = 0x22;
    
    // Transmit the frame
    HAL_StatusTypeDef status = CAN_Transmit(&tx_frame, 100); // 100ms timeout
    
    if (status == HAL_OK) {
        printf("Message transmitted successfully\n");
    } else {
        printf("Transmission failed: %d\n", status);
    }
}
```

### Basic Message Reception:
```c
void can_example_receive(void) {
    CAN_Frame rx_frame;
    
    // Wait for message (blocking)
    HAL_StatusTypeDef status = CAN_Receive(&rx_frame, 1000); // 1 second timeout
    
    if (status == HAL_OK) {
        printf("Received message:\n");
        printf("  ID: 0x%X (%s)\n", 
               rx_frame.id, 
               rx_frame.frame_type == CAN_FRAME_STANDARD ? "Standard" : "Extended");
        printf("  DLC: %d\n", rx_frame.data_length);
        printf("  Data: ");
        for (int i = 0; i < rx_frame.data_length; i++) {
            printf("%02X ", rx_frame.data[i]);
        }
        printf("\n");
        
        if (rx_frame.remote_request) {
            printf("  Remote Request Frame\n");
        }
    } else if (status == HAL_TIMEOUT) {
        printf("No message received (timeout)\n");
    } else {
        printf("Receive error: %d\n", status);
    }
}
```

### Round-Trip Communication:
```c
void can_example_round_trip(void) {
    printf("CAN Round-trip Test\n");
    
    // Send a test message
    CAN_Frame tx_frame = {
        .id = 0x100,
        .frame_type = CAN_FRAME_STANDARD,
        .data_length = 4,
        .remote_request = false
    };
    
    // Fill with test pattern
    tx_frame.data[0] = 0xDE;
    tx_frame.data[1] = 0xAD;
    tx_frame.data[2] = 0xBE;
    tx_frame.data[3] = 0xEF;
    
    printf("Sending: 0x%02X%02X%02X%02X\n",
           tx_frame.data[0], tx_frame.data[1], 
           tx_frame.data[2], tx_frame.data[3]);
    
    CAN_Transmit(&tx_frame, 100);
    
    // Wait a bit for the message to be received
    HAL_Delay(10);
    
    // Try to receive the message
    CAN_Frame rx_frame;
    if (CAN_Receive(&rx_frame, 100) == HAL_OK) {
        printf("Received: 0x%02X%02X%02X%02X\n",
               rx_frame.data[0], rx_frame.data[1], 
               rx_frame.data[2], rx_frame.data[3]);
        
        // Verify data
        bool data_match = true;
        for (int i = 0; i < 4; i++) {
            if (rx_frame.data[i] != tx_frame.data[i]) {
                data_match = false;
                break;
            }
        }
        
        if (data_match) {
            printf("✓ Data integrity verified\n");
        } else {
            printf("✗ Data corruption detected\n");
        }
    } else {
        printf("✗ No response received\n");
    }
}
```

### Mailbox Status Checking:
```c
void can_example_mailbox_status(void) {
    // Check transmit mailbox availability
    if (CAN_IsTransmitMailboxAvailable()) {
        printf("Transmit mailbox available\n");
    } else {
        printf("All transmit mailboxes full\n");
    }
    
    // Check receive FIFO status
    if (CAN_IsReceivePending(0)) {
        uint8_t pending_count = CAN_GetReceivePendingCount(0);
        printf("%d messages pending in FIFO 0\n", pending_count);
    }
    
    if (CAN_IsReceivePending(1)) {
        uint8_t pending_count = CAN_GetReceivePendingCount(1);
        printf("%d messages pending in FIFO 1\n", pending_count);
    }
}
```

### Understanding CAN Timing:
- **Blocking Transmit**: Waits for mailbox availability
- **Non-blocking Transmit**: Returns immediately if no mailbox free
- **Receive Timeout**: How long to wait for incoming messages
- **Bus Arbitration**: Higher priority messages transmit first

<a id='frame-types'></a>
## 6. CAN Frame Types

### Standard vs Extended Frames:
```c
void can_example_frame_types(void) {
    // Standard frame (11-bit ID)
    CAN_Frame std_frame = {
        .id = 0x7FF,                    // Max standard ID
        .frame_type = CAN_FRAME_STANDARD,
        .data_length = 8,
        .remote_request = false
    };
    
    // Extended frame (29-bit ID)
    CAN_Frame ext_frame = {
        .id = 0x1FFFFFFF,               // Max extended ID
        .frame_type = CAN_FRAME_EXTENDED,
        .data_length = 8,
        .remote_request = false
    };
    
    // Fill with test data
    for (int i = 0; i < 8; i++) {
        std_frame.data[i] = i;
        ext_frame.data[i] = i + 10;
    }
    
    printf("Transmitting standard frame (ID: 0x%X)\n", std_frame.id);
    CAN_Transmit(&std_frame, 100);
    
    HAL_Delay(10);
    
    printf("Transmitting extended frame (ID: 0x%X)\n", ext_frame.id);
    CAN_Transmit(&ext_frame, 100);
}
```

### Remote Frames:
```c
void can_example_remote_frames(void) {
    // Send a remote frame to request data
    CAN_Frame remote_frame = {
        .id = 0x200,                    // Request data from node 0x200
        .frame_type = CAN_FRAME_STANDARD,
        .data_length = 8,               // Requested data length
        .remote_request = true          // This is a remote frame
    };
    
    printf("Sending remote frame request for 8 bytes from ID 0x%X\n", remote_frame.id);
    CAN_Transmit(&remote_frame, 100);
    
    // The remote node should respond with a data frame containing the requested data
    // Here we would wait for the response...
}
```

### Variable Data Length:
```c
void can_example_variable_dlc(void) {
    CAN_Frame frame = {
        .id = 0x300,
        .frame_type = CAN_FRAME_STANDARD,
        .remote_request = false
    };
    
    // Test different data lengths
    for (uint8_t dlc = 0; dlc <= 8; dlc++) {
        frame.data_length = dlc;
        
        // Fill data (only up to dlc)
        for (uint8_t i = 0; i < dlc; i++) {
            frame.data[i] = i + 1;
        }
        
        printf("Sending %d bytes: ", dlc);
        for (uint8_t i = 0; i < dlc; i++) {
            printf("%02X ", frame.data[i]);
        }
        printf("\n");
        
        CAN_Transmit(&frame, 100);
        HAL_Delay(50);  // Small delay between transmissions
    }
}
```

### Frame Priority Demonstration:
```c
void can_example_priority(void) {
    // Create frames with different priorities (lower ID = higher priority)
    CAN_Frame high_priority = {
        .id = 0x100,  // High priority (low ID)
        .frame_type = CAN_FRAME_STANDARD,
        .data_length = 4,
        .remote_request = false
    };
    
    CAN_Frame low_priority = {
        .id = 0x200,  // Low priority (high ID)
        .frame_type = CAN_FRAME_STANDARD,
        .data_length = 4,
        .remote_request = false
    };
    
    // Fill with different data
    high_priority.data[0] = 0xAA;
    low_priority.data[0] = 0xBB;
    
    printf("Testing message priority...\n");
    
    // Send both frames quickly
    // The high priority message should be transmitted first
    CAN_Transmit(&low_priority, 10);   // Send low priority first
    CAN_Transmit(&high_priority, 10);  // Then high priority
    
    // In a real bus with contention, the high priority message
    // would win arbitration and be transmitted first
    printf("Both messages queued for transmission\n");
    printf("High priority (ID 0x100) should transmit before low priority (ID 0x200)\n");
}
```

### Frame Error Testing:
```c
void can_example_error_frames(void) {
    // Test with invalid data length
    CAN_Frame invalid_frame = {
        .id = 0x400,
        .frame_type = CAN_FRAME_STANDARD,
        .data_length = 10,  // Invalid! Max is 8
        .remote_request = false
    };
    
    printf("Testing error handling...\n");
    
    // This should fail
    HAL_StatusTypeDef status = CAN_Transmit(&invalid_frame, 100);
    if (status != HAL_OK) {
        printf("✓ Correctly rejected invalid frame (DLC > 8)\n");
    } else {
        printf("✗ Should have rejected invalid frame\n");
    }
    
    // Test with invalid ID for frame type
    CAN_Frame invalid_id_frame = {
        .id = 0x800,  // Invalid for standard frame (> 0x7FF)
        .frame_type = CAN_FRAME_STANDARD,
        .data_length = 4,
        .remote_request = false
    };
    
    status = CAN_Transmit(&invalid_id_frame, 100);
    if (status != HAL_OK) {
        printf("✓ Correctly rejected invalid standard ID\n");
    } else {
        printf("✗ Should have rejected invalid standard ID\n");
    }
}
```

### Frame Statistics:
```c
void can_example_frame_stats(void) {
    CAN_Status status;
    CAN_GetStatus(&status);
    
    printf("CAN Statistics:\n");
    printf("  Initialized: %s\n", status.initialized ? "Yes" : "No");
    printf("  TX Messages: %lu\n", status.tx_count);
    printf("  RX Messages: %lu\n", status.rx_count);
    printf("  Errors: %lu\n", status.error_count);
    printf("  Last Error: ");
    
    switch (status.last_error) {
        case CAN_ERROR_NONE: printf("None\n"); break;
        case CAN_ERROR_STUFF: printf("Stuff Error\n"); break;
        case CAN_ERROR_FORM: printf("Form Error\n"); break;
        case CAN_ERROR_ACK: printf("ACK Error\n"); break;
        case CAN_ERROR_CRC: printf("CRC Error\n"); break;
        case CAN_ERROR_BUS_OFF: printf("Bus Off\n"); break;
        default: printf("Unknown\n"); break;
    }
}
```

<a id='baud-rate'></a>
## 7. Baud Rate Configuration

### Standard Baud Rates:
```c
void can_example_baud_rates(void) {
    // Array of common baud rates to test
    uint32_t baud_rates[] = {
        CAN_BAUD_1000KBPS,  // 1 Mbps
        CAN_BAUD_500KBPS,   // 500 kbps
        CAN_BAUD_250KBPS,   // 250 kbps
        CAN_BAUD_125KBPS,   // 125 kbps
        CAN_BAUD_100KBPS,   // 100 kbps
        CAN_BAUD_50KBPS,    // 50 kbps
        CAN_BAUD_20KBPS,    // 20 kbps
        CAN_BAUD_10KBPS     // 10 kbps
    };
    
    const char* baud_names[] = {
        "1 Mbps", "500 kbps", "250 kbps", "125 kbps",
        "100 kbps", "50 kbps", "20 kbps", "10 kbps"
    };
    
    for (int i = 0; i < 8; i++) {
        printf("Testing %s...\n", baud_names[i]);
        
        // Configure CAN with this baud rate
        CAN_Config config = {
            .mode = CAN_OP_MODE_NORMAL,
            .baud_rate = baud_rates[i],
            .auto_retransmission = true,
            .auto_bus_off_recovery = true,
            .time_triggered_comm = false
        };
        
        HAL_StatusTypeDef status = CAN_Init(&config);
        if (status == HAL_OK) {
            printf("✓ %s configured successfully\n", baud_names[i]);
            
            // Send a test message
            CAN_Frame test_frame = {
                .id = 0x500 + i,  // Different ID for each baud rate
                .frame_type = CAN_FRAME_STANDARD,
                .data_length = 2,
                .remote_request = false
            };
            test_frame.data[0] = i;  // Baud rate index
            test_frame.data[1] = baud_rates[i] & 0xFF;  // LSB of baud rate
            
            CAN_Transmit(&test_frame, 100);
            printf("  Test message sent\n");
            
        } else {
            printf("✗ Failed to configure %s\n", baud_names[i]);
        }
        
        HAL_Delay(100);  // Delay between tests
    }
}
```

### Custom Baud Rate Configuration:
```c
void can_example_custom_baud(void) {
    // Manual bit timing configuration for custom baud rate
    CAN_Config custom_config = {
        .mode = CAN_OP_MODE_NORMAL,
        .baud_rate = 0,  // Custom configuration
        .auto_retransmission = true,
        .auto_bus_off_recovery = true,
        .time_triggered_comm = false,
        .prescaler = 6,        // APB1 clock / 6 = 7MHz CAN clock
        .sync_jump_width = 1,  // 1 time quantum
        .time_segment_1 = 13,  // 13 time quanta
        .time_segment_2 = 2    // 2 time quanta
    };
    
    // Calculate actual baud rate:
    // CAN clock = 42MHz / prescaler = 7MHz
    // Bit time = 1 + 13 + 2 = 16 time quanta
    // Baud rate = 7MHz / 16 = 437.5 kbps
    
    printf("Configuring custom baud rate (437.5 kbps)...\n");
    
    HAL_StatusTypeDef status = CAN_Init(&custom_config);
    if (status == HAL_OK) {
        printf("✓ Custom baud rate configured\n");
        
        // Test with a message
        CAN_Frame frame = {
            .id = 0x600,
            .frame_type = CAN_FRAME_STANDARD,
            .data_length = 4,
            .remote_request = false
        };
        
        sprintf((char*)frame.data, "Hi!");
        CAN_Transmit(&frame, 100);
        
    } else {
        printf("✗ Custom baud rate configuration failed\n");
    }
}
```

### Bit Timing Calculator:
```c
// Function to calculate CAN bit timing parameters
typedef struct {
    uint32_t prescaler;
    uint32_t time_segment_1;
    uint32_t time_segment_2;
    uint32_t sync_jump_width;
} CAN_BitTiming;

bool CAN_CalculateBitTiming(uint32_t target_baud, uint32_t can_clock_hz, CAN_BitTiming* timing) {
    // Common sample points: 75%, 80%, 87.5%
    const float sample_points[] = {0.75f, 0.80f, 0.875f};
    const uint32_t num_sample_points = 3;
    
    for (uint32_t sp_idx = 0; sp_idx < num_sample_points; sp_idx++) {
        float sample_point = sample_points[sp_idx];
        
        // Try different time quanta per bit (8-25 is typical)
        for (uint32_t tq_per_bit = 8; tq_per_bit <= 25; tq_per_bit++) {
            
            // Calculate prescaler
            uint32_t prescaler = can_clock_hz / (target_baud * tq_per_bit);
            if (prescaler < 1 || prescaler > 1024) continue;
            
            // Calculate actual baud rate
            uint32_t actual_baud = can_clock_hz / (prescaler * tq_per_bit);
            if (abs((int)actual_baud - (int)target_baud) > target_baud / 100) continue;  // 1% tolerance
            
            // Calculate time segments
            uint32_t tq_sync = 1;
            uint32_t tq_seg1 = (uint32_t)(sample_point * (tq_per_bit - 1));
            uint32_t tq_seg2 = tq_per_bit - tq_sync - tq_seg1;
            
            // Validate segments
            if (tq_seg1 < 1 || tq_seg1 > 16) continue;
            if (tq_seg2 < 1 || tq_seg2 > 8) continue;
            
            // Success!
            timing->prescaler = prescaler;
            timing->time_segment_1 = tq_seg1;
            timing->time_segment_2 = tq_seg2;
            timing->sync_jump_width = (tq_seg2 > 4) ? 4 : tq_seg2;  // Max 4
            
            return true;
        }
    }
    
    return false;  // No valid timing found
}

void can_example_bit_timing_calc(void) {
    CAN_BitTiming timing;
    
    if (CAN_CalculateBitTiming(500000, 42000000, &timing)) {  // 500kbps, 42MHz clock
        printf("Bit timing for 500kbps:\n");
        printf("  Prescaler: %lu\n", timing.prescaler);
        printf("  Time Segment 1: %lu\n", timing.time_segment_1);
        printf("  Time Segment 2: %lu\n", timing.time_segment_2);
        printf("  Sync Jump Width: %lu\n", timing.sync_jump_width);
        
        // Configure CAN with calculated timing
        CAN_Config config = {
            .mode = CAN_OP_MODE_NORMAL,
            .baud_rate = 0,  // Use custom timing
            .auto_retransmission = true,
            .auto_bus_off_recovery = true,
            .time_triggered_comm = false,
            .prescaler = timing.prescaler,
            .sync_jump_width = timing.sync_jump_width,
            .time_segment_1 = timing.time_segment_1,
            .time_segment_2 = timing.time_segment_2
        };
        
        CAN_Init(&config);
        printf("✓ CAN configured with calculated bit timing\n");
        
    } else {
        printf("✗ Could not calculate valid bit timing\n");
    }
}
```

### Baud Rate Auto-Detection:
```c
void can_example_baud_auto_detect(void) {
    uint32_t possible_bauds[] = {1000000, 500000, 250000, 125000, 100000};
    const char* baud_names[] = {"1Mbps", "500kbps", "250kbps", "125kbps", "100kbps"};
    
    printf("Attempting to auto-detect CAN baud rate...\n");
    
    for (int i = 0; i < 5; i++) {
        printf("Trying %s...\n", baud_names[i]);
        
        // Configure CAN
        CAN_Config config = {
            .mode = CAN_OP_MODE_SILENT,  // Listen-only mode
            .baud_rate = possible_bauds[i],
            .auto_retransmission = false,
            .auto_bus_off_recovery = true,
            .time_triggered_comm = false
        };
        
        CAN_Init(&config);
        
        // Listen for messages for 2 seconds
        uint32_t messages_received = 0;
        uint32_t start_time = HAL_GetTick();
        
        while ((HAL_GetTick() - start_time) < 2000) {
            CAN_Frame frame;
            if (CAN_Receive(&frame, 10) == HAL_OK) {
                messages_received++;
            }
        }
        
        if (messages_received > 0) {
            printf("✓ Detected %s (%lu messages received)\n", 
                   baud_names[i], messages_received);
            break;
        } else {
            printf("✗ No messages at %s\n", baud_names[i]);
        }
    }
}
```

### Baud Rate Switching:
```c
void can_example_baud_switching(void) {
    uint32_t baud_rates[] = {125000, 250000, 500000, 1000000};
    
    printf("Demonstrating baud rate switching...\n");
    
    for (int i = 0; i < 4; i++) {
        printf("Switching to %lu bps...\n", baud_rates[i]);
        
        // Reinitialize CAN with new baud rate
        CAN_Config config = {
            .mode = CAN_OP_MODE_NORMAL,
            .baud_rate = baud_rates[i],
            .auto_retransmission = true,
            .auto_bus_off_recovery = true,
            .time_triggered_comm = false
        };
        
        CAN_Init(&config);
        
        // Send identification message
        CAN_Frame id_frame = {
            .id = 0x700,
            .frame_type = CAN_FRAME_STANDARD,
            .data_length = 4,
            .remote_request = false
        };
        
        id_frame.data[0] = baud_rates[i] & 0xFF;
        id_frame.data[1] = (baud_rates[i] >> 8) & 0xFF;
        id_frame.data[2] = (baud_rates[i] >> 16) & 0xFF;
        id_frame.data[3] = (baud_rates[i] >> 24) & 0xFF;
        
        CAN_Transmit(&id_frame, 100);
        
        HAL_Delay(500);  // Stay at this baud rate for 500ms
    }
    
    printf("Baud rate switching demonstration complete\n");
}
```

<a id='filtering'></a>
## 8. Message Filtering

### Basic Filter Configuration:
```c
void can_example_basic_filter(void) {
    // Configure filter to accept messages with ID 0x100-0x1FF
    CAN_FilterConfig filter = {
        .filter_bank = 0,
        .mode = CAN_FILTER_MODE_ID_MASK,
        .scale = CAN_FILTER_SCALE_32BIT,
        .id = 0x100,     // Accept IDs starting with 0x100
        .mask = 0x700,   // Mask bits 8-10 (0x100-0x1FF range)
        .enable = true
    };
    
    HAL_StatusTypeDef status = CAN_ConfigFilter(&filter);
    if (status == HAL_OK) {
        printf("✓ Filter configured: Accept IDs 0x100-0x1FF\n");
    } else {
        printf("✗ Filter configuration failed\n");
    }
    
    // Test with different message IDs
    uint32_t test_ids[] = {0x050, 0x100, 0x150, 0x1FF, 0x200};
    
    for (int i = 0; i < 5; i++) {
        CAN_Frame test_frame = {
            .id = test_ids[i],
            .frame_type = CAN_FRAME_STANDARD,
            .data_length = 1,
            .remote_request = false
        };
        test_frame.data[0] = i;
        
        CAN_Transmit(&test_frame, 100);
        printf("Sent ID 0x%03X - %s\n", test_ids[i], 
               (test_ids[i] & ~0x700) == 0x100 ? "Should be received" : "Should be filtered");
        
        HAL_Delay(10);
    }
}
```

### Multiple Filter Banks:
```c
void can_example_multiple_filters(void) {
    // Filter bank 0: Accept 0x100-0x1FF
    CAN_FilterConfig filter0 = {
        .filter_bank = 0,
        .mode = CAN_FILTER_MODE_ID_MASK,
        .scale = CAN_FILTER_SCALE_32BIT,
        .id = 0x100,
        .mask = 0x700,
        .enable = true
    };
    
    // Filter bank 1: Accept 0x300-0x3FF
    CAN_FilterConfig filter1 = {
        .filter_bank = 1,
        .mode = CAN_FILTER_MODE_ID_MASK,
        .scale = CAN_FILTER_SCALE_32BIT,
        .id = 0x300,
        .mask = 0x700,
        .enable = true
    };
    
    // Filter bank 2: Accept exact ID 0x500
    CAN_FilterConfig filter2 = {
        .filter_bank = 2,
        .mode = CAN_FILTER_MODE_ID_LIST,
        .scale = CAN_FILTER_SCALE_32BIT,
        .id = 0x500,    // Accept this ID
        .mask = 0x500,  // And this ID (same for single ID)
        .enable = true
    };
    
    CAN_ConfigFilter(&filter0);
    CAN_ConfigFilter(&filter1);
    CAN_ConfigFilter(&filter2);
    
    printf("Multiple filters configured:\n");
    printf("  Bank 0: IDs 0x100-0x1FF\n");
    printf("  Bank 1: IDs 0x300-0x3FF\n");
    printf("  Bank 2: ID 0x500 only\n");
    
    // Test various IDs
    uint32_t test_ids[] = {0x050, 0x123, 0x234, 0x345, 0x456, 0x500, 0x678};
    
    for (int i = 0; i < 7; i++) {
        CAN_Frame frame = {.id = test_ids[i], .frame_type = CAN_FRAME_STANDARD, 
                          .data_length = 1, .remote_request = false};
        frame.data[0] = i;
        CAN_Transmit(&frame, 100);
        
        // Determine if should be accepted
        bool should_accept = false;
        if ((test_ids[i] & ~0x700) == 0x100) should_accept = true;
        if ((test_ids[i] & ~0x700) == 0x300) should_accept = true;
        if (test_ids[i] == 0x500) should_accept = true;
        
        printf("ID 0x%03X: %s\n", test_ids[i], should_accept ? "ACCEPTED" : "FILTERED");
        HAL_Delay(10);
    }
}
```

### 16-bit Filter Configuration:
```c
void can_example_16bit_filters(void) {
    // 16-bit filters for standard IDs only
    CAN_FilterConfig filter = {
        .filter_bank = 3,
        .mode = CAN_FILTER_MODE_ID_MASK,
        .scale = CAN_FILTER_SCALE_16BIT,
        .id = 0x200,     // Base ID
        .mask = 0x7F0,   // Mask for bits 4-10
        .enable = true
    };
    
    CAN_ConfigFilter(&filter);
    printf("16-bit filter: Accept IDs 0x200-0x20F\n");
    
    // Test the filter
    for (uint32_t id = 0x200; id <= 0x210; id++) {
        CAN_Frame frame = {.id = id, .frame_type = CAN_FRAME_STANDARD,
                          .data_length = 1, .remote_request = false};
        frame.data[0] = id & 0xFF;
        CAN_Transmit(&frame, 100);
        
        bool should_accept = (id >= 0x200 && id <= 0x20F);
        printf("ID 0x%03X: %s\n", id, should_accept ? "ACCEPTED" : "FILTERED");
        HAL_Delay(5);
    }
}
```

### Extended ID Filtering:
```c
void can_example_extended_filters(void) {
    // Filter for extended IDs 0x100000-0x100FFF
    CAN_FilterConfig filter = {
        .filter_bank = 4,
        .mode = CAN_FILTER_MODE_ID_MASK,
        .scale = CAN_FILTER_SCALE_32BIT,
        .id = 0x100000,   // 18-bit ID
        .mask = 0x1FFF0000, // Mask upper 13 bits
        .enable = true
    };
    
    CAN_ConfigFilter(&filter);
    printf("Extended ID filter: Accept IDs 0x100000-0x100FFF\n");
    
    // Test with extended frames
    uint32_t test_ids[] = {0x080000, 0x100000, 0x100123, 0x100FFF, 0x101000};
    
    for (int i = 0; i < 5; i++) {
        CAN_Frame frame = {
            .id = test_ids[i],
            .frame_type = CAN_FRAME_EXTENDED,
            .data_length = 2,
            .remote_request = false
        };
        frame.data[0] = i;
        frame.data[1] = test_ids[i] & 0xFF;
        
        CAN_Transmit(&frame, 100);
        
        bool should_accept = (test_ids[i] >= 0x100000 && test_ids[i] <= 0x100FFF);
        printf("Extended ID 0x%06X: %s\n", test_ids[i], 
               should_accept ? "ACCEPTED" : "FILTERED");
        HAL_Delay(10);
    }
}
```

### Filter List Mode:
```c
void can_example_filter_list(void) {
    // Accept only specific IDs: 0x111, 0x222, 0x333
    CAN_FilterConfig filter = {
        .filter_bank = 5,
        .mode = CAN_FILTER_MODE_ID_LIST,
        .scale = CAN_FILTER_SCALE_32BIT,
        .id = 0x111,     // First ID to accept
        .mask = 0x222,   // Second ID to accept
        .enable = true
    };
    
    CAN_ConfigFilter(&filter);
    printf("List filter: Accept only IDs 0x111, 0x222, 0x333\n");
    
    // Note: For 32-bit list mode, we can only specify 2 IDs
    // For more IDs, use multiple filter banks
    
    uint32_t test_ids[] = {0x111, 0x222, 0x333, 0x444, 0x555};
    
    for (int i = 0; i < 5; i++) {
        CAN_Frame frame = {.id = test_ids[i], .frame_type = CAN_FRAME_STANDARD,
                          .data_length = 1, .remote_request = false};
        frame.data[0] = i;
        CAN_Transmit(&frame, 100);
        
        bool should_accept = (test_ids[i] == 0x111 || test_ids[i] == 0x222 || test_ids[i] == 0x333);
        printf("ID 0x%03X: %s\n", test_ids[i], should_accept ? "ACCEPTED" : "FILTERED");
        HAL_Delay(10);
    }
}
```

### Dynamic Filter Reconfiguration:
```c
void can_example_dynamic_filters(void) {
    printf("Dynamic filter reconfiguration demo\n");
    
    // Start with accepting all messages (no filtering)
    // All filter banks disabled by default
    
    printf("Phase 1: Accepting all messages\n");
    for (uint32_t id = 0x100; id <= 0x110; id += 2) {
        CAN_Frame frame = {.id = id, .frame_type = CAN_FRAME_STANDARD,
                          .data_length = 1, .remote_request = false};
        frame.data[0] = id & 0xFF;
        CAN_Transmit(&frame, 100);
        HAL_Delay(5);
    }
    
    // Configure filter to accept only even IDs
    CAN_FilterConfig filter = {
        .filter_bank = 0,
        .mode = CAN_FILTER_MODE_ID_MASK,
        .scale = CAN_FILTER_SCALE_32BIT,
        .id = 0x000,     // Even IDs
        .mask = 0x001,   // Check LSB
        .enable = true
    };
    
    CAN_ConfigFilter(&filter);
    printf("\nPhase 2: Accepting only even IDs\n");
    
    for (uint32_t id = 0x100; id <= 0x110; id += 2) {
        CAN_Frame frame = {.id = id, .frame_type = CAN_FRAME_STANDARD,
                          .data_length = 1, .remote_request = false};
        frame.data[0] = id & 0xFF;
        CAN_Transmit(&frame, 100);
        printf("ID 0x%03X: %s\n", id, (id & 1) == 0 ? "ACCEPTED" : "FILTERED");
        HAL_Delay(5);
    }
    
    // Disable filter
    filter.enable = false;
    CAN_ConfigFilter(&filter);
    printf("\nPhase 3: Filter disabled, accepting all messages\n");
}
```

### Filter Performance Monitoring:
```c
void can_example_filter_performance(void) {
    // Configure a specific filter
    CAN_FilterConfig filter = {
        .filter_bank = 6,
        .mode = CAN_FILTER_MODE_ID_MASK,
        .scale = CAN_FILTER_SCALE_32BIT,
        .id = 0x400,
        .mask = 0x7F0,  // Accept 0x400-0x40F
        .enable = true
    };
    
    CAN_ConfigFilter(&filter);
    
    // Send messages and monitor acceptance
    uint32_t sent_count = 0;
    uint32_t accepted_count = 0;
    
    printf("Testing filter performance...\n");
    
    for (uint32_t id = 0x400; id < 0x420; id++) {
        CAN_Frame tx_frame = {
            .id = id,
            .frame_type = CAN_FRAME_STANDARD,
            .data_length = 1,
            .remote_request = false
        };
        tx_frame.data[0] = sent_count++;
        
        CAN_Transmit(&tx_frame, 100);
        HAL_Delay(2);  // Small delay
        
        // Try to receive (if loopback or another node)
        CAN_Frame rx_frame;
        if (CAN_Receive(&rx_frame, 5) == HAL_OK) {
            accepted_count++;
            printf("Accepted ID 0x%03X\n", rx_frame.id);
        }
    }
    
    printf("Filter performance: %lu/%lu messages accepted\n", accepted_count, sent_count);
}
```

<a id='error-handling'></a>
## 9. Error Handling

### Error Detection and Reporting:
```c
void can_example_error_handling(void) {
    // Register error callback
    CAN_RegisterErrorCallback(can_error_callback);
    
    // Enable error interrupts
    CAN_EnableInterrupts();
    
    printf("CAN error handling enabled\n");
    printf("Monitoring for bus errors...\n");
    
    // Main loop - errors will be reported via callback
    while (1) {
        // Send periodic heartbeat
        CAN_Frame heartbeat = {
            .id = 0x700,
            .frame_type = CAN_FRAME_STANDARD,
            .data_length = 1,
            .remote_request = false
        };
        heartbeat.data[0] = 0xBE;  // "Heartbeat"
        
        CAN_Transmit(&heartbeat, 100);
        
        HAL_Delay(1000);  // Send every second
    }
}

// Error callback function
void can_error_callback(CAN_ErrorType error) {
    printf("CAN Error Detected: ");
    
    switch (error) {
        case CAN_ERROR_STUFF:
            printf("Stuff Error - Bit stuffing violation\n");
            break;
        case CAN_ERROR_FORM:
            printf("Form Error - Frame format violation\n");
            break;
        case CAN_ERROR_ACK:
            printf("ACK Error - No acknowledgment received\n");
            break;
        case CAN_ERROR_BIT_RECESSIVE:
            printf("Bit Recessive Error - Bit level error\n");
            break;
        case CAN_ERROR_BIT_DOMINANT:
            printf("Bit Dominant Error - Bit level error\n");
            break;
        case CAN_ERROR_CRC:
            printf("CRC Error - Frame CRC mismatch\n");
            break;
        case CAN_ERROR_BUS_OFF:
            printf("Bus Off - Node disconnected from bus\n");
            break;
        case CAN_ERROR_BUS_PASSIVE:
            printf("Bus Passive - Error count high, transmission restricted\n");
            break;
        case CAN_ERROR_BUS_WARNING:
            printf("Bus Warning - Error count rising\n");
            break;
        default:
            printf("Unknown error (%d)\n", error);
            break;
    }
    
    // Get detailed error status
    CAN_Status status;
    CAN_GetStatus(&status);
    printf("Error Count: %lu, TX Count: %lu, RX Count: %lu\n",
           status.error_count, status.tx_count, status.rx_count);
}
```

### Bus-Off Recovery:
```c
void can_example_bus_off_recovery(void) {
    printf("Testing bus-off recovery...\n");
    
    // Configure CAN with auto recovery disabled for testing
    CAN_Config config = {
        .mode = CAN_OP_MODE_NORMAL,
        .baud_rate = CAN_BAUD_500KBPS,
        .auto_retransmission = true,
        .auto_bus_off_recovery = false,  // Manual recovery
        .time_triggered_comm = false
    };
    
    CAN_Init(&config);
    
    // Simulate bus errors by transmitting without ACK
    // (This would normally require disconnecting other nodes)
    printf("Attempting to trigger bus-off condition...\n");
    
    // Monitor status
    while (1) {
        CAN_Status status;
        CAN_GetStatus(&status);
        
        if (status.last_error == CAN_ERROR_BUS_OFF) {
            printf("Bus-off condition detected!\n");
            printf("Attempting manual recovery...\n");
            
            // Wait for bus to be idle (11 consecutive recessive bits)
            HAL_Delay(100);
            
            // Reinitialize CAN
            CAN_Init(&config);
            printf("Recovery attempt completed\n");
            break;
        }
        
        HAL_Delay(100);
    }
}
```

### Error Statistics and Monitoring:
```c
void can_example_error_monitoring(void) {
    printf("CAN Error Monitoring\n");
    printf("Monitoring error counters for 30 seconds...\n");
    
    uint32_t start_time = HAL_GetTick();
    CAN_Status initial_status, current_status;
    
    CAN_GetStatus(&initial_status);
    
    while ((HAL_GetTick() - start_time) < 30000) {  // 30 seconds
        CAN_GetStatus(&current_status);
        
        // Check for new errors
        if (current_status.error_count > initial_status.error_count) {
            printf("New errors detected: %lu total\n", current_status.error_count);
            initial_status.error_count = current_status.error_count;
        }
        
        // Report status every 5 seconds
        if (((HAL_GetTick() - start_time) % 5000) < 100) {
            printf("Status: TX=%lu, RX=%lu, Errors=%lu\n",
                   current_status.tx_count,
                   current_status.rx_count,
                   current_status.error_count);
        }
        
        HAL_Delay(100);
    }
    
    printf("Error monitoring complete\n");
    
    // Clear error counters
    CAN_ClearErrors();
    printf("Error counters cleared\n");
}
```

### Fault Injection Testing:
```c
void can_example_fault_injection(void) {
    printf("CAN Fault Injection Testing\n");
    
    // Test 1: ACK Error (no other nodes)
    printf("Test 1: ACK Error\n");
    CAN_Frame frame = {
        .id = 0x100,
        .frame_type = CAN_FRAME_STANDARD,
        .data_length = 1,
        .remote_request = false
    };
    frame.data[0] = 0xAA;
    
    // Send message (should get ACK error if no other nodes)
    HAL_StatusTypeDef status = CAN_Transmit(&frame, 100);
    printf("Transmit status: %d\n", status);
    
    HAL_Delay(1000);
    
    // Test 2: Stuff Error (artificial)
    printf("Test 2: Testing error detection\n");
    // Note: Stuff errors are hard to trigger artificially
    // They occur when 6 consecutive bits of same polarity are sent
    
    // Test 3: Form Error
    printf("Test 3: Form Error testing\n");
    // Form errors occur with invalid frame formats
    
    printf("Fault injection testing complete\n");
}
```

### Error Recovery Strategies:
```c
void can_error_recovery_strategy(CAN_ErrorType error) {
    switch (error) {
        case CAN_ERROR_ACK:
            printf("ACK Error: Check if other nodes are present and powered\n");
            // Strategy: Verify bus connections, check transceiver power
            break;
            
        case CAN_ERROR_CRC:
            printf("CRC Error: Possible electromagnetic interference\n");
            // Strategy: Check cable shielding, reduce baud rate, add filtering
            break;
            
        case CAN_ERROR_BUS_OFF:
            printf("Bus Off: Too many errors, node disconnected\n");
            // Strategy: Wait for bus idle, reinitialize, reduce transmission rate
            CAN_DeInit();
            HAL_Delay(1000);  // Wait for bus recovery
            // Reinitialize with same config
            break;
            
        case CAN_ERROR_BUS_PASSIVE:
            printf("Bus Passive: High error rate, transmission restricted\n");
            // Strategy: Reduce transmission frequency, check bus quality
            break;
            
        case CAN_ERROR_STUFF:
        case CAN_ERROR_FORM:
        case CAN_ERROR_BIT_RECESSIVE:
        case CAN_ERROR_BIT_DOMINANT:
            printf("Physical layer error: Check bus wiring and termination\n");
            // Strategy: Verify termination resistors, check cable integrity
            break;
            
        default:
            printf("Unknown error condition\n");
            break;
    }
}
```

### Robust CAN Communication:
```c
typedef struct {
    uint32_t last_tx_time;
    uint32_t tx_retry_count;
    uint32_t consecutive_errors;
    bool bus_ok;
} CAN_RobustComm;

void can_robust_transmit(CAN_Frame* frame, CAN_RobustComm* comm) {
    const uint32_t MAX_RETRIES = 3;
    const uint32_t ERROR_THRESHOLD = 5;
    const uint32_t HEARTBEAT_INTERVAL = 1000;
    
    uint32_t retries = 0;
    HAL_StatusTypeDef status;
    
    do {
        status = CAN_Transmit(frame, 100);
        
        if (status == HAL_OK) {
            comm->last_tx_time = HAL_GetTick();
            comm->tx_retry_count = 0;
            comm->consecutive_errors = 0;
            comm->bus_ok = true;
            return;
        } else {
            retries++;
            comm->tx_retry_count++;
            comm->consecutive_errors++;
            
            if (comm->consecutive_errors >= ERROR_THRESHOLD) {
                comm->bus_ok = false;
                printf("Bus communication degraded\n");
            }
            
            HAL_Delay(10 * retries);  // Exponential backoff
        }
    } while (retries < MAX_RETRIES);
    
    printf("Failed to transmit after %d retries\n", MAX_RETRIES);
}

void can_robust_monitor(CAN_RobustComm* comm) {
    uint32_t current_time = HAL_GetTick();
    
    // Send periodic heartbeat
    if ((current_time - comm->last_tx_time) > HEARTBEAT_INTERVAL) {
        CAN_Frame heartbeat = {
            .id = 0x7FF,  // High ID, low priority
            .frame_type = CAN_FRAME_STANDARD,
            .data_length = 1,
            .remote_request = false
        };
        heartbeat.data[0] = 0x55;  // Heartbeat pattern
        
        can_robust_transmit(&heartbeat, comm);
    }
    
    // Check bus status
    CAN_Status status;
    CAN_GetStatus(&status);
    
    if (status.last_error != CAN_ERROR_NONE) {
        comm->consecutive_errors++;
        can_error_recovery_strategy(status.last_error);
    }
}
```

<a id='interrupts'></a>
## 10. Interrupt-Driven CAN

### Interrupt Callback Registration:
```c
// Global callback functions
void can_tx_callback(uint8_t mailbox) {
    printf("TX Complete - Mailbox %d\n", mailbox);
}

void can_rx_callback(const CAN_Frame* frame) {
    printf("RX Message: ID=0x%X, DLC=%d\n", frame->id, frame->data_length);
}

void can_error_callback(CAN_ErrorType error) {
    printf("CAN Error: %d\n", error);
}

void can_example_interrupt_setup(void) {
    // Register callbacks
    CAN_RegisterTxCallback(can_tx_callback);
    CAN_RegisterRxCallback(can_rx_callback);
    CAN_RegisterErrorCallback(can_error_callback);
    
    // Enable interrupts
    HAL_StatusTypeDef status = CAN_EnableInterrupts();
    if (status == HAL_OK) {
        printf("CAN interrupts enabled\n");
    } else {
        printf("Failed to enable CAN interrupts\n");
    }
}
```

### Interrupt-Driven Transmission:
```c
#define TX_BUFFER_SIZE 10
CAN_Frame tx_buffer[TX_BUFFER_SIZE];
uint8_t tx_head = 0, tx_tail = 0;

void can_interrupt_transmit(void) {
    // Add frame to transmit buffer
    if (((tx_head + 1) % TX_BUFFER_SIZE) != tx_tail) {
        tx_buffer[tx_head].id = 0x200;
        tx_buffer[tx_head].frame_type = CAN_FRAME_STANDARD;
        tx_buffer[tx_head].data_length = 4;
        tx_buffer[tx_head].remote_request = false;
        sprintf((char*)tx_buffer[tx_head].data, "TX%d", tx_head);
        
        tx_head = (tx_head + 1) % TX_BUFFER_SIZE;
        printf("Frame queued for transmission\n");
    } else {
        printf("TX buffer full\n");
    }
    
    // Try to transmit if mailbox available
    can_process_tx_buffer();
}

void can_process_tx_buffer(void) {
    while ((tx_tail != tx_head) && CAN_IsTransmitMailboxAvailable()) {
        CAN_Transmit(&tx_buffer[tx_tail], 0);  // Non-blocking
        tx_tail = (tx_tail + 1) % TX_BUFFER_SIZE;
    }
}

// TX complete callback processes next frame
void can_tx_complete_callback(uint8_t mailbox) {
    printf("TX mailbox %d completed, processing next frame\n", mailbox);
    can_process_tx_buffer();
}
```

### Interrupt-Driven Reception:
```c
#define RX_BUFFER_SIZE 16
CAN_Frame rx_buffer[RX_BUFFER_SIZE];
uint8_t rx_head = 0, rx_tail = 0;

void can_rx_complete_callback(const CAN_Frame* frame) {
    // Add received frame to buffer
    if (((rx_head + 1) % RX_BUFFER_SIZE) != rx_tail) {
        memcpy(&rx_buffer[rx_head], frame, sizeof(CAN_Frame));
        rx_head = (rx_head + 1) % RX_BUFFER_SIZE;
        printf("Frame received and buffered\n");
    } else {
        printf("RX buffer overflow!\n");
    }
}

void can_process_rx_buffer(void) {
    while (rx_tail != rx_head) {
        CAN_Frame* frame = &rx_buffer[rx_tail];
        
        // Process received frame
        printf("Processing: ID=0x%X, Data=", frame->id);
        for (int i = 0; i < frame->data_length; i++) {
            printf("%02X ", frame->data[i]);
        }
        printf("\n");
        
        rx_tail = (rx_tail + 1) % RX_BUFFER_SIZE;
    }
}

void can_example_interrupt_reception(void) {
    printf("Interrupt-driven reception demo\n");
    
    // Register RX callback
    CAN_RegisterRxCallback(can_rx_complete_callback);
    CAN_EnableInterrupts();
    
    // Main loop processes received frames
    while (1) {
        can_process_rx_buffer();
        HAL_Delay(10);
    }
}
```

### Advanced Interrupt Handling:
```c
typedef struct {
    uint32_t tx_interrupts;
    uint32_t rx_interrupts;
    uint32_t error_interrupts;
    uint32_t fifo0_interrupts;
    uint32_t fifo1_interrupts;
    uint32_t last_interrupt_time;
} CAN_InterruptStats;

CAN_InterruptStats interrupt_stats = {0};

void can_tx_callback_stats(uint8_t mailbox) {
    interrupt_stats.tx_interrupts++;
    interrupt_stats.last_interrupt_time = HAL_GetTick();
    printf("TX Interrupt: Mailbox %d\n", mailbox);
}

void can_rx_callback_stats(const CAN_Frame* frame) {
    interrupt_stats.rx_interrupts++;
    interrupt_stats.last_interrupt_time = HAL_GetTick();
    printf("RX Interrupt: ID=0x%X\n", frame->id);
}

void can_error_callback_stats(CAN_ErrorType error) {
    interrupt_stats.error_interrupts++;
    interrupt_stats.last_interrupt_time = HAL_GetTick();
    printf("Error Interrupt: %d\n", error);
}

void can_fifo0_callback(void) {
    interrupt_stats.fifo0_interrupts++;
    printf("FIFO0 Interrupt\n");
}

void can_fifo1_callback(void) {
    interrupt_stats.fifo1_interrupts++;
    printf("FIFO1 Interrupt\n");
}

void can_example_interrupt_monitoring(void) {
    // Register all callbacks
    CAN_RegisterTxCallback(can_tx_callback_stats);
    CAN_RegisterRxCallback(can_rx_callback_stats);
    CAN_RegisterErrorCallback(can_error_callback_stats);
    
    // Note: FIFO callbacks would need to be added to the CAN driver
    // CAN_RegisterFifo0Callback(can_fifo0_callback);
    // CAN_RegisterFifo1Callback(can_fifo1_callback);
    
    CAN_EnableInterrupts();
    
    printf("Interrupt monitoring started...\n");
    
    uint32_t last_report = HAL_GetTick();
    
    while (1) {
        if ((HAL_GetTick() - last_report) > 5000) {  // Report every 5 seconds
            printf("Interrupt Stats:\n");
            printf("  TX: %lu\n", interrupt_stats.tx_interrupts);
            printf("  RX: %lu\n", interrupt_stats.rx_interrupts);
            printf("  Error: %lu\n", interrupt_stats.error_interrupts);
            printf("  FIFO0: %lu\n", interrupt_stats.fifo0_interrupts);
            printf("  FIFO1: %lu\n", interrupt_stats.fifo1_interrupts);
            printf("  Time since last: %lu ms\n", 
                   HAL_GetTick() - interrupt_stats.last_interrupt_time);
            
            last_report = HAL_GetTick();
        }
        
        HAL_Delay(100);
    }
}
```

### Interrupt Priority Management:
```c
void can_configure_interrupt_priorities(void) {
    // Set CAN interrupt priorities
    HAL_NVIC_SetPriority(CAN1_TX_IRQn, 2, 0);    // TX highest priority
    HAL_NVIC_SetPriority(CAN1_RX0_IRQn, 3, 0);   // RX FIFO0
    HAL_NVIC_SetPriority(CAN1_RX1_IRQn, 3, 0);   // RX FIFO1
    HAL_NVIC_SetPriority(CAN1_SCE_IRQn, 4, 0);   // Error lowest priority
    
    // Enable interrupts
    HAL_NVIC_EnableIRQ(CAN1_TX_IRQn);
    HAL_NVIC_EnableIRQ(CAN1_RX0_IRQn);
    HAL_NVIC_EnableIRQ(CAN1_RX1_IRQn);
    HAL_NVIC_EnableIRQ(CAN1_SCE_IRQn);
    
    printf("CAN interrupt priorities configured\n");
}
```

### Interrupt Context Considerations:
```c
// Variables that may be accessed from interrupt context
volatile uint32_t interrupt_counter = 0;
volatile bool data_ready = false;
volatile CAN_Frame latest_frame;

// Safe interrupt callback
void can_rx_callback_thread_safe(const CAN_Frame* frame) {
    // Copy data quickly
    memcpy((void*)&latest_frame, frame, sizeof(CAN_Frame));
    data_ready = true;
    interrupt_counter++;
    
    // Don't do heavy processing here!
    // Keep interrupt handlers short
}

// Main thread processes the data
void can_process_interrupt_data(void) {
    if (data_ready) {
        data_ready = false;
        
        // Now we can do heavy processing
        printf("Processing frame from interrupt: ID=0x%X\n", latest_frame.id);
        
        // Process the frame data...
        for (int i = 0; i < latest_frame.data_length; i++) {
            // Heavy processing here is OK
            process_data_byte(latest_frame.data[i]);
        }
    }
}
```

### Low-Power Interrupt Operation:
```c
void can_enter_low_power_mode(void) {
    printf("Entering low-power mode with CAN wake-up\n");
    
    // Configure CAN to wake from sleep on message reception
    CAN_Config config = {
        .mode = CAN_OP_MODE_NORMAL,
        .baud_rate = CAN_BAUD_125KBPS,  // Lower power at lower speed
        .auto_retransmission = false,
        .auto_bus_off_recovery = true,
        .time_triggered_comm = false
    };
    
    CAN_Init(&config);
    
    // Enable wake-up interrupt
    __HAL_CAN_ENABLE_IT(&hcan1, CAN_IT_WKU);
    
    // Enter sleep mode
    HAL_SuspendTick();
    HAL_PWR_EnterSLEEPMode(PWR_MAINREGULATOR_ON, PWR_SLEEPENTRY_WFI);
    HAL_ResumeTick();
    
    printf("Woke up from CAN message\n");
}
```

<a id='multi-node'></a>
## 11. Multi-Node Networks

### Node Identification and Addressing:
```c
#define NODE_ID_ENGINE        0x10
#define NODE_ID_TRANSMISSION  0x11
#define NODE_ID_BRAKES        0x12
#define NODE_ID_BODY          0x13
#define NODE_ID_INFO          0x14

#define MSG_HEARTBEAT        0x100
#define MSG_ENGINE_SPEED     0x200
#define MSG_VEHICLE_SPEED    0x201
#define MSG_FUEL_LEVEL       0x202
#define MSG_BRAKE_PRESSURE   0x300
#define MSG_DOOR_STATUS      0x400
#define MSG_LIGHT_STATUS     0x401

typedef struct {
    uint8_t node_id;
    const char* node_name;
    uint32_t heartbeat_interval;
    uint32_t last_heartbeat;
} CAN_Node;

CAN_Node local_node = {
    .node_id = NODE_ID_ENGINE,
    .node_name = "Engine ECU",
    .heartbeat_interval = 1000,
    .last_heartbeat = 0
};
```

### Heartbeat and Node Monitoring:
```c
void can_send_heartbeat(void) {
    uint32_t current_time = HAL_GetTick();
    
    if ((current_time - local_node.last_heartbeat) >= local_node.heartbeat_interval) {
        CAN_Frame heartbeat = {
            .id = MSG_HEARTBEAT | local_node.node_id,
            .frame_type = CAN_FRAME_STANDARD,
            .data_length = 4,
            .remote_request = false
        };
        
        // Pack heartbeat data
        heartbeat.data[0] = local_node.node_id;
        heartbeat.data[1] = (current_time >> 24) & 0xFF;
        heartbeat.data[2] = (current_time >> 16) & 0xFF;
        heartbeat.data[3] = (current_time >> 8) & 0xFF;
        
        CAN_Transmit(&heartbeat, 100);
        local_node.last_heartbeat = current_time;
        
        printf("Heartbeat sent from %s\n", local_node.node_name);
    }
}

typedef struct {
    uint8_t node_id;
    uint32_t last_seen;
    bool active;
} NodeStatus;

#define MAX_NODES 8
NodeStatus node_status[MAX_NODES];

void can_monitor_nodes(void) {
    uint32_t current_time = HAL_GetTick();
    
    for (int i = 0; i < MAX_NODES; i++) {
        if (node_status[i].active) {
            uint32_t time_since_last = current_time - node_status[i].last_seen;
            
            if (time_since_last > 3000) {  // 3 seconds timeout
                printf("Node %d offline (last seen %lu ms ago)\n", 
                       node_status[i].node_id, time_since_last);
                node_status[i].active = false;
            }
        }
    }
}

void can_process_heartbeat(const CAN_Frame* frame) {
    if ((frame->id & 0xFF00) == MSG_HEARTBEAT) {
        uint8_t node_id = frame->id & 0xFF;
        
        // Find or create node status
        for (int i = 0; i < MAX_NODES; i++) {
            if (node_status[i].node_id == node_id || !node_status[i].active) {
                node_status[i].node_id = node_id;
                node_status[i].last_seen = HAL_GetTick();
                node_status[i].active = true;
                printf("Node %d heartbeat received\n", node_id);
                break;
            }
        }
    }
}
```

### Data Broadcasting and Subscription:
```c
typedef struct {
    uint32_t message_id;
    void (*callback)(const CAN_Frame* frame);
    bool active;
} CAN_Subscription;

#define MAX_SUBSCRIPTIONS 16
CAN_Subscription subscriptions[MAX_SUBSCRIPTIONS];

void can_subscribe(uint32_t message_id, void (*callback)(const CAN_Frame* frame)) {
    for (int i = 0; i < MAX_SUBSCRIPTIONS; i++) {
        if (!subscriptions[i].active) {
            subscriptions[i].message_id = message_id;
            subscriptions[i].callback = callback;
            subscriptions[i].active = true;
            printf("Subscribed to message 0x%X\n", message_id);
            break;
        }
    }
}

void can_unsubscribe(uint32_t message_id) {
    for (int i = 0; i < MAX_SUBSCRIPTIONS; i++) {
        if (subscriptions[i].active && subscriptions[i].message_id == message_id) {
            subscriptions[i].active = false;
            printf("Unsubscribed from message 0x%X\n", message_id);
            break;
        }
    }
}

void can_process_subscriptions(const CAN_Frame* frame) {
    for (int i = 0; i < MAX_SUBSCRIPTIONS; i++) {
        if (subscriptions[i].active && subscriptions[i].message_id == frame->id) {
            if (subscriptions[i].callback) {
                subscriptions[i].callback(frame);
            }
        }
    }
}
```

### Engine ECU Example:
```c
void engine_speed_callback(const CAN_Frame* frame) {
    if (frame->data_length >= 2) {
        uint16_t rpm = (frame->data[0] << 8) | frame->data[1];
        printf("Engine speed: %d RPM\n", rpm);
    }
}

void engine_ecu_init(void) {
    local_node.node_id = NODE_ID_ENGINE;
    local_node.node_name = "Engine ECU";
    
    // Subscribe to relevant messages
    can_subscribe(MSG_VEHICLE_SPEED, vehicle_speed_callback);
    can_subscribe(MSG_FUEL_LEVEL, fuel_level_callback);
    
    // Configure filters to accept relevant messages
    CAN_FilterConfig filter = {
        .filter_bank = 0,
        .mode = CAN_FILTER_MODE_ID_MASK,
        .scale = CAN_FILTER_SCALE_32BIT,
        .id = 0x200,     // Accept 0x200-0x2FF (vehicle data)
        .mask = 0x700,
        .enable = true
    };
    CAN_ConfigFilter(&filter);
    
    printf("Engine ECU initialized\n");
}

void engine_ecu_main(void) {
    static uint32_t last_engine_data = 0;
    uint32_t current_time = HAL_GetTick();
    
    // Send engine data every 100ms
    if ((current_time - last_engine_data) >= 100) {
        CAN_Frame engine_frame = {
            .id = MSG_ENGINE_SPEED,
            .frame_type = CAN_FRAME_STANDARD,
            .data_length = 2,
            .remote_request = false
        };
        
        // Simulate engine speed (1000-6000 RPM)
        uint16_t rpm = 1000 + (rand() % 5000);
        engine_frame.data[0] = (rpm >> 8) & 0xFF;
        engine_frame.data[1] = rpm & 0xFF;
        
        CAN_Transmit(&engine_frame, 50);
        last_engine_data = current_time;
    }
    
    // Send heartbeat
    can_send_heartbeat();
    
    // Monitor other nodes
    can_monitor_nodes();
}
```

### Vehicle Network Simulation:
```c
void can_network_simulation(void) {
    printf("CAN Network Simulation\n");
    printf("Simulating a vehicle network with multiple ECUs\n");
    
    // Initialize different node types
    engine_ecu_init();
    // transmission_ecu_init();
    // brakes_ecu_init();
    // body_ecu_init();
    
    // Register message processing
    CAN_RegisterRxCallback(can_message_router);
    
    CAN_EnableInterrupts();
    
    while (1) {
        // Each ECU runs its main loop
        engine_ecu_main();
        // transmission_ecu_main();
        // brakes_ecu_main();
        // body_ecu_main();
        
        HAL_Delay(10);
    }
}

void can_message_router(const CAN_Frame* frame) {
    // Route messages to appropriate handlers
    can_process_heartbeat(frame);
    can_process_subscriptions(frame);
    
    // Route based on message ID ranges
    if ((frame->id & 0xFF00) == 0x2000) {
        // Vehicle data messages
        // process_vehicle_data(frame);
    } else if ((frame->id & 0xFF00) == 0x3000) {
        // Brake system messages
        // process_brake_data(frame);
    } else if ((frame->id & 0xFF00) == 0x4000) {
        // Body control messages
        // process_body_data(frame);
    }
}
```

### Network Diagnostics:
```c
typedef struct {
    uint32_t total_messages;
    uint32_t messages_per_second;
    uint32_t error_count;
    uint32_t node_count;
    uint32_t last_measurement_time;
} CAN_NetworkStats;

CAN_NetworkStats network_stats = {0};

void can_update_network_stats(const CAN_Frame* frame) {
    network_stats.total_messages++;
    
    uint32_t current_time = HAL_GetTick();
    if ((current_time - network_stats.last_measurement_time) >= 1000) {
        network_stats.messages_per_second = network_stats.total_messages - 
                                          (network_stats.messages_per_second * 
                                           (current_time - network_stats.last_measurement_time) / 1000);
        network_stats.last_measurement_time = current_time;
    }
}

void can_network_diagnostics(void) {
    printf("\n=== CAN Network Diagnostics ===\n");
    printf("Total Messages: %lu\n", network_stats.total_messages);
    printf("Messages/sec: %lu\n", network_stats.messages_per_second);
    printf("Error Count: %lu\n", network_stats.error_count);
    printf("Active Nodes: %lu\n", network_stats.node_count);
    
    // List active nodes
    printf("Active Nodes:\n");
    for (int i = 0; i < MAX_NODES; i++) {
        if (node_status[i].active) {
            printf("  Node %d: Last seen %lu ms ago\n", 
                   node_status[i].node_id, 
                   HAL_GetTick() - node_status[i].last_seen);
        }
    }
    
    // Check bus status
    CAN_Status can_status;
    CAN_GetStatus(&can_status);
    printf("CAN Status: TX=%lu, RX=%lu, Errors=%lu\n",
           can_status.tx_count, can_status.rx_count, can_status.error_count);
}
```

### Fault-Tolerant Multi-Node Communication:
```c
typedef struct {
    uint32_t message_id;
    uint8_t expected_sender;
    uint32_t timeout_ms;
    uint32_t last_received;
    bool timeout_reported;
} CAN_MessageMonitor;

#define MAX_MONITORED_MESSAGES 10
CAN_MessageMonitor monitored_messages[MAX_MONITORED_MESSAGES];

void can_monitor_message(uint32_t message_id, uint8_t expected_sender, uint32_t timeout_ms) {
    for (int i = 0; i < MAX_MONITORED_MESSAGES; i++) {
        if (monitored_messages[i].message_id == 0) {  // Empty slot
            monitored_messages[i].message_id = message_id;
            monitored_messages[i].expected_sender = expected_sender;
            monitored_messages[i].timeout_ms = timeout_ms;
            monitored_messages[i].last_received = HAL_GetTick();
            monitored_messages[i].timeout_reported = false;
            break;
        }
    }
}

void can_check_message_timeouts(void) {
    uint32_t current_time = HAL_GetTick();
    
    for (int i = 0; i < MAX_MONITORED_MESSAGES; i++) {
        if (monitored_messages[i].message_id != 0) {
            uint32_t time_since_last = current_time - monitored_messages[i].last_received;
            
            if (time_since_last > monitored_messages[i].timeout_ms) {
                if (!monitored_messages[i].timeout_reported) {
                    printf("WARNING: Message 0x%X from node %d timed out (%lu ms)\n",
                           monitored_messages[i].message_id,
                           monitored_messages[i].expected_sender,
                           time_since_last);
                    monitored_messages[i].timeout_reported = true;
                }
            }
        }
    }
}

void can_message_received(const CAN_Frame* frame) {
    // Update monitoring for received messages
    for (int i = 0; i < MAX_MONITORED_MESSAGES; i++) {
        if (monitored_messages[i].message_id == frame->id) {
            monitored_messages[i].last_received = HAL_GetTick();
            monitored_messages[i].timeout_reported = false;
            break;
        }
    }
    
    can_update_network_stats(frame);
}
```

<a id='monitoring'></a>
## 12. CAN Bus Monitoring

### Passive Monitoring Mode:
```c
void can_enter_monitor_mode(void) {
    printf("Entering CAN bus monitor mode...\n");
    
    // Configure CAN for silent mode (listen-only)
    CAN_Config config = {
        .mode = CAN_OP_MODE_SILENT,
        .baud_rate = CAN_BAUD_500KBPS,
        .auto_retransmission = false,  // Don't transmit
        .auto_bus_off_recovery = true,
        .time_triggered_comm = false
    };
    
    HAL_StatusTypeDef status = CAN_Init(&config);
    if (status != HAL_OK) {
        printf("Failed to enter monitor mode\n");
        return;
    }
    
    // Disable all filters to receive all messages
    // (All filters disabled by default accepts all)
    
    printf("Monitor mode active. Listening for all CAN traffic...\n");
    
    // Register callback to log all received messages
    CAN_RegisterRxCallback(can_monitor_callback);
    CAN_EnableInterrupts();
}

void can_monitor_callback(const CAN_Frame* frame) {
    static uint32_t message_count = 0;
    message_count++;
    
    // Log message details
    printf("[%lu] ", HAL_GetTick());
    
    if (frame->frame_type == CAN_FRAME_EXTENDED) {
        printf("EXT ");
    } else {
        printf("STD ");
    }
    
    printf("ID:0x%08X DLC:%d DATA:", frame->id, frame->data_length);
    
    for (int i = 0; i < frame->data_length; i++) {
        printf("%02X ", frame->data[i]);
    }
    
    if (frame->remote_request) {
        printf("[RTR]");
    }
    
    printf("\n");
    
    // Periodic statistics
    if (message_count % 100 == 0) {
        printf("--- %lu messages received ---\n", message_count);
    }
}
```

### Bus Load Analysis:
```c
typedef struct {
    uint32_t start_time;
    uint32_t bit_count;
    uint32_t message_count;
    float bus_load_percentage;
} CAN_BusLoadAnalyzer;

CAN_BusLoadAnalyzer bus_analyzer = {0};

void can_start_bus_load_analysis(void) {
    bus_analyzer.start_time = HAL_GetTick();
    bus_analyzer.bit_count = 0;
    bus_analyzer.message_count = 0;
    
    printf("Starting bus load analysis...\n");
}

void can_analyze_message(const CAN_Frame* frame) {
    bus_analyzer.message_count++;
    
    // Calculate bits in this message
    // SOF (1) + ID (11/29) + RTR (1) + IDE (1) + r0 (1) + DLC (4) + 
    // Data (0-64) + CRC (15) + ACK (2) + EOF (7) + IFS (3) = ~47-111 bits
    uint32_t bits_in_message = 47;  // Base overhead
    
    if (frame->frame_type == CAN_FRAME_EXTENDED) {
        bits_in_message += 18;  // Extended ID additional bits
    }
    
    bits_in_message += frame->data_length * 8;  // Data bits
    bits_in_message += 15;  // CRC
    bits_in_message += 2;   // ACK
    bits_in_message += 7;   // EOF
    bits_in_message += 3;   // IFS
    
    bus_analyzer.bit_count += bits_in_message;
}

void can_report_bus_load(void) {
    uint32_t elapsed_ms = HAL_GetTick() - bus_analyzer.start_time;
    if (elapsed_ms == 0) return;
    
    uint32_t elapsed_seconds = elapsed_ms / 1000;
    if (elapsed_seconds == 0) return;
    
    // Calculate bus load
    // Assume 500kbps = 500,000 bits/second
    uint32_t baud_rate = 500000;  // 500 kbps
    uint64_t total_bits_possible = (uint64_t)baud_rate * elapsed_seconds;
    uint64_t total_bits_used = bus_analyzer.bit_count;
    
    bus_analyzer.bus_load_percentage = (float)total_bits_used / total_bits_possible * 100.0f;
    
    printf("Bus Load Analysis (%lu seconds):\n", elapsed_seconds);
    printf("  Messages: %lu\n", bus_analyzer.message_count);
    printf("  Total bits: %lu\n", bus_analyzer.bit_count);
    printf("  Bus load: %.1f%%\n", bus_analyzer.bus_load_percentage);
    printf("  Messages/sec: %.1f\n", (float)bus_analyzer.message_count / elapsed_seconds);
    
    if (bus_analyzer.bus_load_percentage > 80.0f) {
        printf("  WARNING: High bus load!\n");
    }
}
```

### Message ID Analysis:
```c
#define MAX_UNIQUE_IDS 256
typedef struct {
    uint32_t message_id;
    uint32_t count;
    uint32_t last_seen;
} CAN_ID_Stats;

CAN_ID_Stats id_stats[MAX_UNIQUE_IDS];
uint32_t unique_id_count = 0;

void can_analyze_message_ids(const CAN_Frame* frame) {
    // Find existing ID or add new one
    for (uint32_t i = 0; i < unique_id_count; i++) {
        if (id_stats[i].message_id == frame->id) {
            id_stats[i].count++;
            id_stats[i].last_seen = HAL_GetTick();
            return;
        }
    }
    
    // New ID
    if (unique_id_count < MAX_UNIQUE_IDS) {
        id_stats[unique_id_count].message_id = frame->id;
        id_stats[unique_id_count].count = 1;
        id_stats[unique_id_count].last_seen = HAL_GetTick();
        unique_id_count++;
    }
}

void can_report_id_analysis(void) {
    printf("\nMessage ID Analysis:\n");
    printf("Total unique IDs: %lu\n", unique_id_count);
    printf("\nTop 10 most frequent IDs:\n");
    
    // Simple bubble sort by count (for small arrays)
    for (uint32_t i = 0; i < unique_id_count - 1; i++) {
        for (uint32_t j = 0; j < unique_id_count - i - 1; j++) {
            if (id_stats[j].count < id_stats[j + 1].count) {
                CAN_ID_Stats temp = id_stats[j];
                id_stats[j] = id_stats[j + 1];
                id_stats[j + 1] = temp;
            }
        }
    }
    
    for (uint32_t i = 0; i < 10 && i < unique_id_count; i++) {
        printf("  0x%08X: %lu messages\n", 
               id_stats[i].message_id, id_stats[i].count);
    }
}
```

### Error Rate Monitoring:
```c
typedef struct {
    uint32_t start_time;
    uint32_t error_count;
    uint32_t message_count;
    float error_rate;  // Errors per 1000 messages
} CAN_ErrorRateMonitor;

CAN_ErrorRateMonitor error_monitor = {0};

void can_start_error_monitoring(void) {
    error_monitor.start_time = HAL_GetTick();
    error_monitor.error_count = 0;
    error_monitor.message_count = 0;
    error_monitor.error_rate = 0.0f;
    
    printf("Starting error rate monitoring...\n");
}

void can_monitor_errors(CAN_ErrorType error) {
    error_monitor.error_count++;
    
    printf("CAN Error detected: ");
    switch (error) {
        case CAN_ERROR_ACK: printf("ACK Error\n"); break;
        case CAN_ERROR_CRC: printf("CRC Error\n"); break;
        case CAN_ERROR_STUFF: printf("Stuff Error\n"); break;
        case CAN_ERROR_FORM: printf("Form Error\n"); break;
        case CAN_ERROR_BUS_OFF: printf("Bus Off\n"); break;
        default: printf("Other Error (%d)\n", error); break;
    }
}

void can_update_error_rate(const CAN_Frame* frame) {
    error_monitor.message_count++;
    
    if (error_monitor.message_count >= 100) {
        error_monitor.error_rate = (float)error_monitor.error_count / 
                                  error_monitor.message_count * 1000.0f;
    }
}

void can_report_error_rate(void) {
    uint32_t elapsed_seconds = (HAL_GetTick() - error_monitor.start_time) / 1000;
    
    printf("\nError Rate Report (%lu seconds):\n", elapsed_seconds);
    printf("  Total messages: %lu\n", error_monitor.message_count);
    printf("  Total errors: %lu\n", error_monitor.error_count);
    printf("  Error rate: %.2f errors per 1000 messages\n", error_monitor.error_rate);
    printf("  Error percentage: %.2f%%\n", 
           (float)error_monitor.error_count / error_monitor.message_count * 100.0f);
    
    if (error_monitor.error_rate > 10.0f) {
        printf("  WARNING: High error rate!\n");
    }
}
```

### Network Topology Discovery:
```c
typedef struct {
    uint8_t node_id;
    char node_type[20];
    uint32_t capabilities;
    bool active;
    uint32_t last_seen;
} CAN_NetworkNode;

#define MAX_NETWORK_NODES 16
CAN_NetworkNode network_nodes[MAX_NETWORK_NODES];

void can_discover_network_topology(void) {
    printf("Discovering network topology...\n");
    
    // Send discovery message
    CAN_Frame discovery = {
        .id = 0x7F0,  // Discovery message ID
        .frame_type = CAN_FRAME_STANDARD,
        .data_length = 2,
        .remote_request = false
    };
    discovery.data[0] = local_node.node_id;
    discovery.data[1] = 0x01;  // Discovery request
    
    CAN_Transmit(&discovery, 100);
    
    // Listen for responses for 5 seconds
    uint32_t start_time = HAL_GetTick();
    uint32_t discovered_count = 0;
    
    while ((HAL_GetTick() - start_time) < 5000) {
        CAN_Frame response;
        if (CAN_Receive(&response, 100) == HAL_OK) {
            if (response.id == 0x7F1 && response.data_length >= 4) {  // Discovery response
                uint8_t node_id = response.data[0];
                uint8_t node_type = response.data[1];
                uint32_t capabilities = (response.data[2] << 8) | response.data[3];
                
                // Add to network nodes
                for (int i = 0; i < MAX_NETWORK_NODES; i++) {
                    if (!network_nodes[i].active || network_nodes[i].node_id == node_id) {
                        network_nodes[i].node_id = node_id;
                        network_nodes[i].capabilities = capabilities;
                        network_nodes[i].last_seen = HAL_GetTick();
                        network_nodes[i].active = true;
                        
                        // Determine node type
                        switch (node_type) {
                            case 1: strcpy(network_nodes[i].node_type, "Engine ECU"); break;
                            case 2: strcpy(network_nodes[i].node_type, "Transmission"); break;
                            case 3: strcpy(network_nodes[i].node_type, "Brakes"); break;
                            case 4: strcpy(network_nodes[i].node_type, "Body Control"); break;
                            default: strcpy(network_nodes[i].node_type, "Unknown"); break;
                        }
                        
                        discovered_count++;
                        break;
                    }
                }
            }
        }
    }
    
    printf("Network discovery complete. Found %d nodes:\n", discovered_count);
    for (int i = 0; i < MAX_NETWORK_NODES; i++) {
        if (network_nodes[i].active) {
            printf("  Node %d: %s (0x%04X)\n", 
                   network_nodes[i].node_id,
                   network_nodes[i].node_type,
                   network_nodes[i].capabilities);
        }
    }
}
```

### Performance Benchmarking:
```c
typedef struct {
    uint32_t messages_sent;
    uint32_t messages_received;
    uint32_t transmit_time_min;
    uint32_t transmit_time_max;
    uint32_t transmit_time_avg;
    uint32_t receive_latency_min;
    uint32_t receive_latency_max;
    uint32_t receive_latency_avg;
} CAN_PerformanceStats;

CAN_PerformanceStats perf_stats = {0};

void can_performance_test(void) {
    printf("Starting CAN performance test...\n");
    
    // Reset stats
    memset(&perf_stats, 0, sizeof(perf_stats));
    perf_stats.transmit_time_min = UINT32_MAX;
    perf_stats.receive_latency_min = UINT32_MAX;
    
    // Test transmit performance
    const int TEST_MESSAGES = 1000;
    
    for (int i = 0; i < TEST_MESSAGES; i++) {
        CAN_Frame frame = {
            .id = 0x100 + (i % 256),
            .frame_type = CAN_FRAME_STANDARD,
            .data_length = 8,
            .remote_request = false
        };
        
        // Fill with test data
        for (int j = 0; j < 8; j++) {
            frame.data[j] = (i + j) & 0xFF;
        }
        
        uint32_t start_time = HAL_GetTick();
        HAL_StatusTypeDef status = CAN_Transmit(&frame, 10);  // Short timeout
        uint32_t end_time = HAL_GetTick();
        
        if (status == HAL_OK) {
            uint32_t transmit_time = end_time - start_time;
            perf_stats.messages_sent++;
            
            if (transmit_time < perf_stats.transmit_time_min) 
                perf_stats.transmit_time_min = transmit_time;
            if (transmit_time > perf_stats.transmit_time_max) 
                perf_stats.transmit_time_max = transmit_time;
            perf_stats.transmit_time_avg += transmit_time;
        }
        
        HAL_Delay(1);  // Small delay between messages
    }
    
    if (perf_stats.messages_sent > 0) {
        perf_stats.transmit_time_avg /= perf_stats.messages_sent;
    }
    
    printf("Performance Test Results:\n");
    printf("  Messages sent: %lu/%d\n", perf_stats.messages_sent, TEST_MESSAGES);
    printf("  Transmit time - Min: %lu ms, Max: %lu ms, Avg: %lu ms\n",
           perf_stats.transmit_time_min, perf_stats.transmit_time_max, 
           perf_stats.transmit_time_avg);
    printf("  Transmit rate: %.1f msg/sec\n", 
           (float)perf_stats.messages_sent / ((HAL_GetTick() - (HAL_GetTick() - TEST_MESSAGES)) / 1000.0f));
}
```